In [ ]:
import numpy as np
import pandas as pd
import os
import glob

In [1]:
from platform import python_version print(python_version())

SyntaxError: invalid syntax (795133577.py, line 1)

In [ ]:
os.getcwd()

In [ ]:
os.chdir('C:\\Users\\...')
os.listdir()

In [ ]:
# data_files is set to all .csv files in the cwd
data_files = glob.glob('*.csv')
# df will be concatenation of the csv files (including assigning the source file as an extra column for every row) 
# in data_files and the originating index will not carried in
df = pd.concat([pd.read_csv(filename).assign(source=filename) for filename in data_files], ignore_index=True)
# print the number of rows and columns
print(df.shape)

In [ ]:
# enable visualisation of the entire df using code 
# pd.set_option('display.max_rows', None)
# for now just view the head, including column names
df.head()

In [ ]:
# and view the last 5 rows, note the sample ID and Sample ID are considered different
df.tail()

In [ ]:
# when a cell in the column 'sample ID' is Nan, fill it with the corresponding cell in 'Sample ID'
df['sample ID'] = df['sample ID'].fillna(df['Sample ID'])
# view the head again
df.head()

In [ ]:
# and view the tail again
df.tail()

In [ ]:
# create a series called sample_ID_nan_count and check how many Nan values remain in'sample ID' column
sample_ID_nan_count = df['sample ID'].isna().sum()
print(sample_ID_nan_count)

In [ ]:
# sample_ID_nan_count = 0 so all sample IDs have been captured, therefore delete the column 'Sample ID' (capital S)
df = df.drop('Sample ID', axis=1)
df.head()

In [ ]:
# use the column header 'Sample Number' from now on
df = df.rename(columns={'sample ID': 'Sample Number'})
df.head()

In [ ]:
# this saves the df to the cwd, disabled to prevent a million copies of the .csv affecting the original import!
# df.to_csv('HGPgeno_data_one_sample_number.csv',index=False)

In [ ]:
# some of these columns are not useful so want to identify what pandas recognises as thier name to then remove them
column_names = list(df.columns.values)
print(column_names)

In [ ]:
#.drop will remove columns or rows
# axis = 1 tells pandas to look for a column, axis = 0 is the default and is for rows
# The default of .drop is to return a new DataFrame instead of modifying the existing DataFrame.
# Setting inplace = True means edits are made to the existing 'df_concatsource'
df.drop(['MN01G', 'MN02G', 'MN03G', 'MN04G', 'MN05G', 'KE01G', 'KE02G', 'KE03G', 'KE04G', 'KE06G', 'KE07G',
                      'JK01G', 'JK02G', 'DO01G', 'DO02G', 'FY01G', 'FY02G', 'FYGATA', 'GYPB230', 'JS', 'DO', 'MN_AOD',
                      'S_AOD', 'K', 'KP', 'GATA', 'GYPBint5', 'JK', 'FY'], axis=1, inplace=True)
df.head()

In [ ]:
# removing the maximum number of columns visible enables human visualisation that only RH columns, Sample Number 
# and source columns remain
pd.set_option('display.max_columns', None)
df.head()

In [ ]:
# again, saved just the once
# df.to_csv('HGPgeno_data_one_sample_number_rh_only.csv',index=False)

In [ ]:
# many duplicates will exist within the dataset
duplicate_Sample_Number_geno_data = df['Sample Number'].duplicated()
print(duplicate_Sample_Number_geno_data.value_counts())

In [ ]:
# Many duplicates will be due to re-test due to incomplete genotype data
# Any row with incomplete RH genotype calls will now be deleted
# some of these genotypes are incmoplete and left blank in the source, meaning they are now 'NaN' in df
# count the number of nan values in each column
print(df.isnull().sum())

In [ ]:
# other incomplete genotypes contain 'UND'
# this code uses get
countund_RHD_INT4 = df['RHD_INT4'].value_counts().get('UND', 0)
print("Occurrences of 'UND':", countund_RHD_INT4)

In [ ]:
# this code uses sum and is easier to understand
sumund_RHD_INT4 = (df['RHD_INT4'] == 'UND').sum()
print("Occurrences of 'UND':", sumund_RHD_INT4)

In [ ]:
sumund_RHD455 = (df['RHD455'] == 'UND').sum()
print("Occurrences of 'UND':", sumund_RHD455)

In [ ]:
sumund_RHD667 = (df['RHD667'] == 'UND').sum()
print("Occurrences of 'UND':", sumund_RHD667)

In [ ]:
sumund_EX5 = (df['EX5'] == 'UND').sum()
print("Occurrences of 'UND':", sumund_EX5)

In [ ]:
# need to use pandas to find a row index (or maybe all rows) with UND

In [ ]:
# to see an example row with UND in genotypes
print(df.loc[[5377]])

In [ ]:
# unfortunately 'UND' also exists in phenotype columns that have 
# what migth be genuine genotype results that are not translated 
print(df.loc[[208]])

In [ ]:
# rows with UND in phenotype columns and NOT in genotype coumns must be kept
# similarly, rows exist with NaN in a phenotype but with complete genotypes - again these rows must be retained
# it is therefore not possible to simply replace 'UND' throughout the df with NaN and delete all rows with Nan

In [ ]:
# tried some solutions in this cell but a bit confusing or don't work - retained just for now
# df[df.apply(lambda col: col.str.contains('UND')).any(axis=1)]
# df.head()
# df_rhd455und = df[df['RHD455'].str.contains('UND')]
#, ['RHD_INT4'].str.contains('UND'), ['RHD667'].str.contains('UND'), ['EX5'].str.contains('UND'), ['RHC'].str.contains('UND'), ['RH_EX7'].str.contains('UND'), ['RHE'].str.contains('UND'), ['RHCE667'].str.contains('UND'), ['RHCE1006'].str.contains('UND'), ['RHCE712'].str.contains('UND'), 'RHCE733'].str.contains('UND')]



In [ ]:
# Convert cells with UND, in genotype columns, to 'NaN'.
df['RHD_INT4'] = df['RHD_INT4'].replace('UND', np.nan)

In [ ]:
print(df.loc[[5377]])

In [ ]:
sumund_RHD_INT4 = (df['RHD_INT4'] == 'UND').sum()
print("Occurrences of 'UND':", sumund_RHD_INT4)
# was 5 before conversion to UND

In [ ]:
print(df.isnull().sum())
# was 19 in RHD_INT4 before conversion of UND to NaN, 5 + 19 = 24

In [ ]:
print(df.shape)

In [ ]:
# delete rows with NaN values in specific columns
# should be 8132 - 24 = 8108
df.dropna(subset=['RHD_INT4'], inplace=True)
print(df.shape)

In [ ]:
# need to print the number of NaN in each column again as the total in each column may have changed by deleteing 
# all rows with NaN in RHD_INT4 i.e. some rows will have had NaN in mulitple columns
print(df.isnull().sum())

In [ ]:
# Convert cells with UND, in genotype columns, to 'NaN'.
df['RHD455'] = df['RHD455'].replace('UND', np.nan)
df['RHD667'] = df['RHD667'].replace('UND', np.nan)
df['EX5'] = df['EX5'].replace('UND', np.nan)
df['RHC'] = df['RHC'].replace('UND', np.nan)
df['RH_EX7'] = df['RH_EX7'].replace('UND', np.nan)
df['RHE'] = df['RHE'].replace('UND', np.nan)
df['RHCE667'] = df['RHCE667'].replace('UND', np.nan)
df['RHCE1006'] = df['RHCE1006'].replace('UND', np.nan)
df['RHCE712'] = df['RHCE712'].replace('UND', np.nan)
df['RHCE733'] = df['RHCE733'].replace('UND', np.nan)

In [ ]:
print(df.isnull().sum())

In [ ]:
# delete rows with NaN values in specific columns
df.dropna(subset=['RHD455'], inplace=True)
df.dropna(subset=['RHD667'], inplace=True)
df.dropna(subset=['EX5'], inplace=True)
df.dropna(subset=['RHC'], inplace=True)
df.dropna(subset=['RH_EX7'], inplace=True)
df.dropna(subset=['RHE'], inplace=True)
df.dropna(subset=['RHCE667'], inplace=True)
df.dropna(subset=['RHCE1006'], inplace=True)
df.dropna(subset=['RHCE712'], inplace=True)
df.dropna(subset=['RHCE733'], inplace=True)
print(df.shape)
# NOTE: new shape may not be 8108-(9+0+9+013+47+209+31+7+11+146+41+0) [incidentally = 7594] 
# as many NaN values in different columns will appear in the same rows

In [ ]:
print(df.isnull().sum())

In [ ]:
# NOTE TO SELF: Do I really want to delete all RHCE712 UND rows?

In [ ]:
# fill all NaN results with UND - this can be done across the entire df now that
# NaN in genotypes has been removed
df.fillna('UND', inplace = True)

In [ ]:
print(df.isnull().all())

In [ ]:
# Now there are no NaN in the entire df
# what other values are hiding in there?
counts = df['RH01G'].value_counts()
print(counts)

In [ ]:
# to use value_counts on multiple columns at once
# df.apply(pd.Series.value_counts)
# doesn't work!

In [ ]:
counts = df['RH02G'].value_counts()
print(counts)

In [ ]:
counts = df['RH03G'].value_counts()
print(counts)

In [ ]:
counts = df['RH04G'].value_counts()
print(counts)

In [ ]:
counts = df['RH05G'].value_counts()
print(counts)

In [ ]:
# the difference between ? and UND is an artefact of the software versions used in NHSBT
# NEXT STEP: Transform all phenotype results of ? to UND
df = df.replace('?', 'UND')

In [ ]:
counts = df['RH01G'].value_counts()
print(counts)
# in RH01G this is 102 + 38 = 140
# genotype data and phenotype data is now clean and usable

In [ ]:
print(df.shape)

In [ ]:
# need now to examine the Sample Number column
# count the number of duplicated Sample Numbers and call it 'duplicate_Sample_Number_geno_data' and print the value
duplicate_Sample_Number_geno_data = df['Sample Number'].duplicated()
print(duplicate_Sample_Number_geno_data.value_counts())

In [ ]:
df.head()

In [ ]:
# identify duplicated sample numbers
# .duplicated finds duplicates, ['Sample Number'] ensures Pandas finds duplicates only in this column,
# "keep=False" returns rows with duplicates in the specified column in the data set below.
# Returning a slice of the df dataframe allows visulaisation of the rows identified
# adding .sort_values('Sample Number'), organises the slice by ascending sample no. to enable easier human visualisation
df[df.duplicated('Sample Number', keep=False) == True].sort_values('Sample Number')

In [ ]:
# NEXT STEPS: identify all rows in Sample Number column where value does not start &d099
# ~ acts as boolean for NOT
noamperd099 = df[~df["Sample Number"].str.contains('&d099*')]
print(noamperd099)

In [ ]:
# will convert '&D099' to '&d099'.  NOTE need to use str.replace, not simply .replace as above
# df.str.replace('*D099*', '*d099*')

df['Sample Number'] = df['Sample Number'].str.replace('&D099*', '&d099*')

In [ ]:
# return series of all rows where Sample Number is not &d099
noamperd099 = df[~df["Sample Number"].str.contains('&d099*')]
print(noamperd099)
# doesn't work - don't know why

In [ ]:
# edit that series so that letters in all strings are changed to lower case
# df['Sample Number'] = df['Sample Number'].str.lower

In [ ]:
# noamperd099 = df[~df["Sample Number"].str.contains('&d099*')]
# print(noamperd099)
# RETURNS ERROR 
# Moving on

In [ ]:
df.head()

In [ ]:
df = df.drop([656, 657, 2230, 6095, 6135, 6155, 6177, 6197])
print(df.shape)

In [ ]:
# reduced by 8 as 8 rows deleted

In [ ]:
df[df.duplicated('Sample Number', keep=False) == True].sort_values('Sample Number')

In [ ]:
# duplicates reduced by 4 because the 4 #NAME have been removed
# the &D099 and G numbers were not duplicates

In [ ]:
# As only complete Genotypes have been retained, duplicates can be deleted so as to retain one only
# even if controls are kept, they will be excluded (presumably) at merge with PID
# it is possible that the same sample number has been tested twice, or duplicated in the original dataset,
# and the two rows have identical results
# remove rows that are identical across specific columns, this is required as the column labelled "source" 
# will likely be different
df.drop_duplicates(subset=['Sample Number', 'RH01G', 'RH02G', 'RH03G', 'RH04G', 'RH05G', 'RH10G', 'RH20G', 'RH19G', 'RH31G', 'RHD455',
                                 'RHD_INT4', 'RHD667',	'EX5', 'RHC', 'RH_EX7', 'RHE', 'RHCE667', 'RHCE1006',	'RHCE712', 'RHCE733'],
                         keep='first', inplace=True)
print(df.shape)
# 7621 - 6670 = 951
# NOTE keep='first' is default but included specifically as 951 exact duplicate rows is a lot!

In [ ]:
df[df.duplicated('Sample Number', keep=False) == True].sort_values('Sample Number')
# this slice shows 24 pairs and 1 quad that duplicate on Sampl Number but have different genotype results

In [ ]:
# given the number is relatively small specific rows will be manually deleted
# rows with NOAMP results that resolve on the duplicate test will be deleted
df = df.drop([349, 1523, 2694, 2711, 3782, 7222, 8081, 7789])
print(df.shape)

In [ ]:
df[df.duplicated('Sample Number', keep=False) == True].sort_values('Sample Number')
# this slice returns 18 pairs

In [ ]:
# this data contains the ame samples tested years apart - why?
# samples with different genotype results - which is correct?
# This data will be dropped
df_dupsfordrop = df[df.duplicated('Sample Number', keep=False) == True].sort_values('Sample Number')
list(df_dupsfordrop.index.values)

In [ ]:
df = df.drop(labels=[487,
 688,
 534,
 691,
 333,
 700,
 5268,
 1880,
 5269,
 1893,
 5270,
 1898,
 5271,
 1945,
 2041,
 759,
 2736,
 764,
 1980,
 1408,
 1996,
 2050,
 938,
 1440,
 4457,
 4409,
 4425,
 4456,
 4875,
 4433,
 4447,
 4876,
 4954,
 5004,
 5937,
 6270], axis=0)
print(df.shape)
# 6662 - 36 = 6626

In [ ]:
# trim &d from all sample numbers
df['Sample Number'] = df['Sample Number'].str.replace('&d', '')
df.head(20)

In [ ]:
# these sample numbers have the two digit code for check digits
# Sample Numbers in the df_PID have the one digit check digit
# e.g. in this data set 099131500109087 is 0991315001090Q in df_PID
# the check digits are important for human data entry, not so much for computer
# will trim -1 in df_PID 
# will trim -2 in df here
df['Sample Number'] = df['Sample Number'].str[:-2]

In [ ]:
df.head()

In [ ]:
# df is now ready for merge with df_PID (created below)

In [ ]:
# df.to_csv('HGPgeno_data_ready_for_merge.csv',index=False)